In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("MP-DE").getOrCreate()

In [0]:
df = spark.read.table('cl_mp_de.`01_bronze`.bronze_sales')
display(df)

### Standardizing column names

In [0]:
df_standardized = df.withColumnRenamed('Trxn_ID#', 'transaction_id') \
                    .withColumnRenamed('!Date_Ref!', 'date') \
                    .withColumnRenamed('PROD_CODE_ID', 'product_id') \
                    .withColumnRenamed('Cust_ID_99', 'customer_id') \
                    .withColumnRenamed('Store_Loc_ID', 'store_id') \
                    .withColumnRenamed('Qty_Sold', 'quantity_sold') \
                    .withColumnRenamed('_Unit_Price_', 'unit_price')
display(df_standardized)

In [0]:
from pyspark.sql.functions import col, coalesce, date_format, initcap, trim, expr, when, regexp_replace, to_date

#standardizing date format to 'yyyy-MM-dd'
#casting date (string to date)
df_clean = df_standardized.withColumn(
        "date",
        coalesce(
            expr("try_to_date(trim(`date`), 'dd-MM-yy')"),
            expr("try_to_date(trim(`date`), 'dd-MM-yyyy')"),
            expr("try_to_date(initcap(trim(`date`)), 'dd-MMM-yy')"),
            expr("try_to_date(initcap(trim(`date`)), 'dd-MMM-yyyy')"),
            expr("try_to_date(trim(`date`), 'yyyy.MM.dd')"),
            expr("try_to_date(trim(`date`), 'yyyyMMdd')"),
            expr("try_to_date(trim(`date`), 'MM-dd-yyyy')")
        ).cast('date')
    )

#NULL in customer_id is replaced with "Unknown"
df_clean = df_clean.withColumn(
        'customer_id',
        when(col('customer_id').isin("NULL"), "Unknown").otherwise(col('customer_id'))
    )

#unit_price is casted to double
df_clean =  df_clean.withColumn(
        'unit_price',
        regexp_replace(col('unit_price'), '[^0-9.]', '').cast('double')
    )

#quantity_sold is casted to integer
df_clean = df_clean.withColumn(
        'quantity_sold', col('quantity_sold').cast('int')
    )
 
display(df_clean)

In [0]:
df_clean.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('cl_mp_de.`02_silver`.silver_sales')